# Phase 4 (CJPE) - data prep
Pull gated HF data, clean, audit, analyse, tokenise with the Phase-1 tokenizer, build folds.

**Before running:** Internet ON, Kaggle secret `HF_TOKEN` (or `HF`), attach the Phase-1 tokenizer dataset.

**Output:** `/kaggle/working/cjpe_data_clean/`

## 1. Configuration

In [ ]:
import os
import re
import sys
import json
import glob
import time
import pickle
import shutil
import hashlib
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SPLITS = ["train", "dev", "test"]

# Env vars are only for local testing. On Kaggle leave them alone.
LOCAL_DATA_DIR = os.environ.get("CJPE_LOCAL_DATA_DIR", None)
TOKENIZER_DIR = os.environ.get("CJPE_TOKENIZER_DIR", None)
OUT_DIR = os.environ.get("CJPE_OUT_DIR", "/kaggle/working/cjpe_data_clean")

SEED = 42
N_FOLDS = 5

# Cleaning
KEEP_NEWLINES = True        # keep paragraph breaks (useful for section analysis)
EMPTY_PLACEHOLDER = "."     # used for an empty test text (a test case is never dropped)
SHORT_DOC_CHARS = 200       # only a warning

# Long-document views (only used for the summary tables and the self-check here)
CHUNK_LEN = 1024            # tokens per chunk, incl. the final SEP
CHUNK_OVERLAP = 128
MAX_CHUNKS = 16
HEAD_TAIL_LEN = 1024
HEAD_FRAC = 0.5

# Quick text baselines (CPU) used to see which part of a judgment carries signal
BASELINE_MAX_TRAIN = 5000
BASELINE_MAX_DEV = 3000

os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)
print("OUT_DIR:", OUT_DIR)

## 2. Hugging Face login

In [ ]:
HF_TOKEN = None

if LOCAL_DATA_DIR is None:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in ["HF_TOKEN", "HF"]:
            if HF_TOKEN is None:
                try:
                    HF_TOKEN = secrets.get_secret(secret_name)
                    print("Using Kaggle secret:", secret_name)
                except Exception:
                    pass
    except ImportError:
        print("kaggle_secrets not available (not on Kaggle).")

    if HF_TOKEN is None:
        HF_TOKEN = os.environ.get("HF_TOKEN", None)

    if HF_TOKEN is None:
        raise RuntimeError("No HF token found. Add a Kaggle secret named HF_TOKEN (or HF).")

    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("Local data mode, skipping Hugging Face login.")

## 3. Find and download the CJPE files (train / dev / test)

In [ ]:
RAW_DIR = os.path.join(OUT_DIR, "raw_snapshot")
os.makedirs(RAW_DIR, exist_ok=True)


def simple_name(path):
    name = path.lower()
    name = name.replace("_", "")
    name = name.replace("-", "")
    name = name.replace(" ", "")
    return name


candidates = {"train": [], "dev": [], "test": []}

if LOCAL_DATA_DIR is not None:
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for file_name in files:
            stem, ext = os.path.splitext(file_name.lower())
            if ext not in [".json", ".jsonl"]:
                continue
            if stem in candidates:
                candidates[stem].append(os.path.join(root, file_name))
else:
    from huggingface_hub import list_repo_files
    all_files = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    print("Files in repo:", len(all_files))
    for path in all_files:
        key = simple_name(path)
        if ("cjpe" not in key) and ("phase4" not in key):
            continue
        stem, ext = os.path.splitext(os.path.basename(path).lower())
        if ext not in [".json", ".jsonl"]:
            continue
        if stem in candidates:
            candidates[stem].append(path)

for split in SPLITS:
    print(split, "->", candidates[split])

for split in SPLITS:
    if len(candidates[split]) == 0:
        print("\nAll repo files, to help you spot the right ones:")
        if LOCAL_DATA_DIR is None:
            for p in all_files:
                print("  ", p)
        raise FileNotFoundError("No file found for split: " + split)

SPLIT_PATH = {}
for split in SPLITS:
    options = sorted(candidates[split], key=len)
    chosen = options[0]
    if len(options) > 1:
        print("WARNING: several candidates for", split, "- using the shortest path:", chosen)
    if LOCAL_DATA_DIR is not None:
        local_path = chosen
    else:
        from huggingface_hub import hf_hub_download
        local_path = hf_hub_download(HF_DATASET_REPO, chosen, repo_type="dataset", token=HF_TOKEN)
    ext = os.path.splitext(local_path)[1]
    saved_copy = os.path.join(RAW_DIR, split + ext)
    shutil.copy(local_path, saved_copy)
    SPLIT_PATH[split] = saved_copy
    print(split, "ready:", saved_copy, "|", round(os.path.getsize(saved_copy) / 1e6, 1), "MB")

## 4. Load (tolerant to several JSON layouts)

In [ ]:
ID_KEYS = ["id", "case_id", "caseid", "doc_id", "ID", "Id"]
TEXT_KEYS = ["text", "document", "case_text", "judgment_text", "content", "Text"]
LABEL_KEYS = ["label", "labels", "judgment", "outcome", "decision", "y", "Label"]
LIST_KEYS = ["data", "examples", "records", "cases", "documents", "items"]


def read_any(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        raw = f.read().strip()
    if raw == "":
        return []
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        rows = []
        for line in raw.splitlines():
            line = line.strip()
            if line == "":
                continue
            rows.append(json.loads(line))
        return rows


def first_present(record, keys):
    for k in keys:
        if k in record:
            return k
    return None


def to_record_list(obj):
    # Returns a list of (id, record_dict_or_text)
    out = []
    if isinstance(obj, dict):
        found_list = None
        for k in LIST_KEYS:
            if k in obj and isinstance(obj[k], list):
                found_list = obj[k]
        if found_list is not None:
            obj = found_list
        else:
            for key, value in obj.items():
                out.append((key, value))
            return out
    for i in range(len(obj)):
        out.append((None, obj[i]))
    return out


def label_to_int(value):
    # Returns 0 or 1, or -1 if the label is missing / unusable.
    if value is None:
        return -1
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        if value == 0:
            return 0
        if value == 1:
            return 1
        return -1
    if isinstance(value, str):
        v = value.strip()
        if v in ["0", "0.0"]:
            return 0
        if v in ["1", "1.0"]:
            return 1
    return -1


raw_docs = {}
for split in SPLITS:
    obj = read_any(SPLIT_PATH[split])
    pairs = to_record_list(obj)
    docs = []
    bad_label_values = Counter()
    for pos in range(len(pairs)):
        key, rec = pairs[pos]
        if isinstance(rec, str):
            doc_id = key
            text = rec
            label_raw = None
        else:
            id_key = first_present(rec, ID_KEYS)
            text_key = first_present(rec, TEXT_KEYS)
            label_key = first_present(rec, LABEL_KEYS)
            if id_key is not None:
                doc_id = rec[id_key]
            elif key is not None:
                doc_id = key
            else:
                doc_id = "doc_" + str(pos)
            if text_key is None:
                raise KeyError("No text field in record. Keys seen: " + str(list(rec.keys())))
            text = rec[text_key]
            label_raw = rec[label_key] if label_key is not None else None
        label = label_to_int(label_raw)
        if split != "test" and label == -1:
            bad_label_values[str(label_raw)] += 1
        if split == "test":
            label = -1
        docs.append({"id": doc_id, "text": text, "label": label})
    raw_docs[split] = docs
    print(split, "| docs:", len(docs), "| unusable labels:", dict(bad_label_values))

if len(raw_docs["test"]) == 0:
    print("WARNING: test file is empty right now. The spec says data is dynamic - re-run later.")
if len(raw_docs["train"]) == 0:
    raise RuntimeError("train file has 0 docs.")
print("\nFirst train record keys check:")
print({"id": raw_docs["train"][0]["id"], "label": raw_docs["train"][0]["label"], "text[:150]": str(raw_docs["train"][0]["text"])[:150]})

## 5. Clean

In [ ]:
# Build the character map from code numbers only (keeps this cell pure ASCII).
CHAR_MAP = {}
for cp in range(0, 9):
    CHAR_MAP[cp] = None            # control chars: delete
CHAR_MAP[9] = " "                  # tab
CHAR_MAP[11] = " "                 # vertical tab
CHAR_MAP[12] = " "                 # form feed
for cp in range(14, 32):
    CHAR_MAP[cp] = None
for cp in range(127, 160):
    CHAR_MAP[cp] = None
for cp in [0x200B, 0x200C, 0x200D, 0x2060, 0xFEFF, 0x00AD, 0xFFFD]:
    CHAR_MAP[cp] = None            # zero-width chars, soft hyphen, replacement char: delete
SPACE_CODES = [0x00A0, 0x1680, 0x202F, 0x205F, 0x3000]
for cp in range(0x2000, 0x200B):
    SPACE_CODES.append(cp)
for cp in SPACE_CODES:
    CHAR_MAP[cp] = " "             # odd spaces -> normal space
CHAR_MAP[0x2028] = "\n"            # line / paragraph separators -> newline
CHAR_MAP[0x2029] = "\n"

MANY_SPACES = re.compile(r" {2,}")
MANY_NEWLINES = re.compile(r"\n{3,}")


def clean_text(text):
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = text.translate(CHAR_MAP)
    lines = text.split("\n")
    cleaned_lines = []
    for line in lines:
        line = MANY_SPACES.sub(" ", line)
        line = line.strip()
        cleaned_lines.append(line)
    text = "\n".join(cleaned_lines)
    if KEEP_NEWLINES:
        text = MANY_NEWLINES.sub("\n\n", text)
    else:
        text = text.replace("\n", " ")
        text = MANY_SPACES.sub(" ", text)
    text = text.strip()
    return text


clean_docs = {}
drop_log = []
clean_stats = {"changed_docs": 0, "chars_removed": 0}

for split in SPLITS:
    kept = []
    seen_ids = set()
    for d in raw_docs[split]:
        original_text = d["text"] if isinstance(d["text"], str) else str(d["text"])
        text = clean_text(original_text)
        if text != original_text:
            clean_stats["changed_docs"] += 1
            clean_stats["chars_removed"] += max(0, len(original_text) - len(text))
        id_str = str(d["id"])
        if split != "test":
            if d["label"] == -1:
                drop_log.append((split, d["id"], "unusable label"))
                continue
            if text == "":
                drop_log.append((split, d["id"], "empty text"))
                continue
            if id_str in seen_ids:
                drop_log.append((split, d["id"], "duplicate id"))
                continue
        else:
            if text == "":
                text = EMPTY_PLACEHOLDER
                print("test doc with empty text kept with placeholder:", d["id"])
            if id_str in seen_ids:
                print("WARNING: duplicate id inside test (kept, check with TAs):", d["id"])
        seen_ids.add(id_str)
        kept.append({"id": d["id"], "text": text, "label": d["label"]})
    clean_docs[split] = kept
    print(split, "raw:", len(raw_docs[split]), "-> kept:", len(kept))

print("\nCleaning touched", clean_stats["changed_docs"], "docs, removed", clean_stats["chars_removed"], "chars")
print("Dropped so far:", len(drop_log))
for item in drop_log[:15]:
    print("  ", item)

## 6. Audit: duplicates, conflicts, leakage

In [ ]:
def text_hash(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def head_hash(text, n=500):
    return hashlib.sha1(text[:n].lower().encode("utf-8")).hexdigest()


# 6a. exact duplicate texts inside train+dev (train first, so train copy wins)
seen_hash = {}
conflict_hashes = set()
for split in ["train", "dev"]:
    for d in clean_docs[split]:
        h = text_hash(d["text"])
        if h in seen_hash:
            if seen_hash[h] != d["label"]:
                conflict_hashes.add(h)
        else:
            seen_hash[h] = d["label"]

dup_removed = 0
conflict_removed = 0
emitted = set()
for split in ["train", "dev"]:
    kept = []
    for d in clean_docs[split]:
        h = text_hash(d["text"])
        if h in conflict_hashes:
            drop_log.append((split, d["id"], "same text with conflicting labels"))
            conflict_removed += 1
            continue
        if h in emitted:
            drop_log.append((split, d["id"], "exact duplicate text"))
            dup_removed += 1
            continue
        emitted.add(h)
        kept.append(d)
    clean_docs[split] = kept
print("exact duplicates removed:", dup_removed, "| conflicting-label copies removed:", conflict_removed)

# 6b. test vs train/dev overlap (report only, test is never dropped)
labelled_hashes = set()
labelled_head_hashes = set()
for split in ["train", "dev"]:
    for d in clean_docs[split]:
        labelled_hashes.add(text_hash(d["text"]))
        labelled_head_hashes.add(head_hash(d["text"]))

test_exact = 0
test_head = 0
for d in clean_docs["test"]:
    if text_hash(d["text"]) in labelled_hashes:
        test_exact += 1
    if head_hash(d["text"]) in labelled_head_hashes:
        test_head += 1
print("test docs whose exact text is also in train/dev:", test_exact, "of", len(clean_docs["test"]))
print("test docs whose first 500 chars match a train/dev doc:", test_head)

# 6c. very short docs, id overlap across splits
short_counts = {}
for split in SPLITS:
    n_short = 0
    for d in clean_docs[split]:
        if len(d["text"]) < SHORT_DOC_CHARS:
            n_short += 1
    short_counts[split] = n_short
print("docs shorter than", SHORT_DOC_CHARS, "chars:", short_counts)

train_ids = set()
for d in clean_docs["train"]:
    train_ids.add(str(d["id"]))
dev_ids = set()
for d in clean_docs["dev"]:
    dev_ids.add(str(d["id"]))
test_ids = set()
for d in clean_docs["test"]:
    test_ids.add(str(d["id"]))
print("id overlap train/dev:", len(train_ids & dev_ids), "| train/test:", len(train_ids & test_ids), "| dev/test:", len(dev_ids & test_ids))

for split in SPLITS:
    print(split, "final docs:", len(clean_docs[split]))

audit = {
    "raw_counts": {s: len(raw_docs[s]) for s in SPLITS},
    "final_counts": {s: len(clean_docs[s]) for s in SPLITS},
    "dropped": [list(map(str, x)) for x in drop_log],
    "exact_dup_removed": dup_removed,
    "conflict_removed": conflict_removed,
    "test_exact_in_labelled": test_exact,
    "test_head_match_labelled": test_head,
    "short_docs": short_counts,
    "clean_stats": clean_stats,
}

## 7. Text analysis: labels, lengths, sections

In [ ]:
rows = []
for split in SPLITS:
    for d in clean_docs[split]:
        text = d["text"]
        words = text.split()
        paras = 0
        for p in text.split("\n"):
            if p.strip() != "":
                paras += 1
        rows.append({
            "split": split,
            "id": str(d["id"]),
            "label": d["label"],
            "n_chars": len(text),
            "n_words": len(words),
            "n_paras": paras,
        })
df_all = pd.DataFrame(rows)

print("=== Label distribution ===")
for split in ["train", "dev"]:
    sub = df_all[df_all["split"] == split]
    counts = sub["label"].value_counts().sort_index()
    total = len(sub)
    line = split + ": "
    for lab in counts.index:
        line += "label " + str(lab) + "=" + str(int(counts[lab])) + " (" + str(round(100.0 * counts[lab] / total, 1)) + "%)  "
    print(line)

lab_df = df_all[df_all["split"].isin(["train", "dev"])]
n0 = int((lab_df["label"] == 0).sum())
n1 = int((lab_df["label"] == 1).sum())
print("train+dev: label0 =", n0, "| label1 =", n1, "| imbalance ratio (major/minor) =", round(max(n0, n1) / max(1, min(n0, n1)), 2))

print("\n=== Length stats (words) by split ===")
print(df_all.groupby("split")["n_words"].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

print("\n=== Length stats (words) by label, train+dev ===")
print(lab_df.groupby("label")["n_words"].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

from sklearn.metrics import roc_auc_score
if lab_df["label"].nunique() == 2:
    auc_len = roc_auc_score(lab_df["label"], lab_df["n_words"])
    print("\nAUC of raw length alone for predicting label 1:", round(auc_len, 3), "(0.5 = no length shortcut)")

In [ ]:
# Outcome words near the end of the judgment (which label do they go with?)
TAIL_CHARS = 1500
patterns = [
    "dismissed", "allowed", "partly allowed", "set aside", "upheld", "affirmed",
    "rejected", "accepted", "no order as to costs", "appeal is allowed", "appeal is dismissed",
]
tail_rows = []
for pat in patterns:
    n_with = 0
    n_with_label1 = 0
    for split in ["train", "dev"]:
        for d in clean_docs[split]:
            tail = d["text"][-TAIL_CHARS:].lower()
            if pat in tail:
                n_with += 1
                if d["label"] == 1:
                    n_with_label1 += 1
    if n_with > 0:
        p1 = n_with_label1 / n_with
    else:
        p1 = float("nan")
    tail_rows.append({"phrase_in_last_%d_chars" % TAIL_CHARS: pat, "docs": n_with, "P(label=1 | phrase)": round(p1, 3)})
print(pd.DataFrame(tail_rows).to_string(index=False))
print("\nIf P is near 0 or 1 for common phrases the tail leaks the outcome - keep the tail in your input.")
print("If P is near the base rate, the outcome text was removed from the document.")

In [ ]:
# Which part of the judgment carries signal? TF-IDF + logistic regression on one slice at a time.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


def make_segment(words, name):
    n = len(words)
    if n == 0:
        return "."
    if name == "full":
        return " ".join(words)
    if name == "first_512_words":
        return " ".join(words[:512])
    if name == "last_512_words":
        return " ".join(words[-512:])
    if name == "head256_tail256":
        if n <= 512:
            return " ".join(words)
        return " ".join(words[:256] + words[-256:])
    if name.startswith("part_"):
        k = int(name.split("_")[1])
        start = int(n * (k - 1) / 5)
        end = int(n * k / 5)
        piece = words[start:end]
        if len(piece) == 0:
            return "."
        return " ".join(piece)
    raise ValueError(name)


train_docs = clean_docs["train"]
dev_docs = clean_docs["dev"]
rng = np.random.RandomState(SEED)

if len(train_docs) > BASELINE_MAX_TRAIN:
    idx = rng.choice(len(train_docs), BASELINE_MAX_TRAIN, replace=False)
    base_train = []
    for i in idx:
        base_train.append(train_docs[int(i)])
else:
    base_train = train_docs

if len(dev_docs) > BASELINE_MAX_DEV:
    idx = rng.choice(len(dev_docs), BASELINE_MAX_DEV, replace=False)
    base_dev = []
    for i in idx:
        base_dev.append(dev_docs[int(i)])
else:
    base_dev = dev_docs

print("baseline train docs:", len(base_train), "| dev docs:", len(base_dev))

train_words = []
for d in base_train:
    train_words.append(d["text"].split())
dev_words = []
for d in base_dev:
    dev_words.append(d["text"].split())
y_tr = np.array([d["label"] for d in base_train])
y_dv = np.array([d["label"] for d in base_dev])

segment_names = ["full", "first_512_words", "last_512_words", "head256_tail256",
                 "part_1", "part_2", "part_3", "part_4", "part_5"]
baseline_rows = []

if len(base_dev) > 0 and len(set(y_tr.tolist())) == 2:
    for name in segment_names:
        t0 = time.time()
        tr_texts = []
        for w in train_words:
            tr_texts.append(make_segment(w, name))
        dv_texts = []
        for w in dev_words:
            dv_texts.append(make_segment(w, name))
        vec = TfidfVectorizer(max_features=50000, sublinear_tf=True, min_df=2)
        X_tr = vec.fit_transform(tr_texts)
        X_dv = vec.transform(dv_texts)
        clf = LogisticRegression(max_iter=1000, class_weight="balanced")
        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_dv)
        f1 = f1_score(y_dv, pred, average="macro")
        auc = float("nan")
        if len(set(y_dv.tolist())) == 2:
            auc = roc_auc_score(y_dv, clf.predict_proba(X_dv)[:, 1])
        baseline_rows.append({"slice": name, "macro_F1": round(f1, 4), "AUC": round(auc, 4), "sec": round(time.time() - t0, 1)})
        print(name, "macro-F1:", round(f1, 4), "AUC:", round(auc, 4))

    # length only
    len_tr = np.log1p(np.array([[len(w)] for w in train_words]))
    len_dv = np.log1p(np.array([[len(w)] for w in dev_words]))
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(len_tr, y_tr)
    pred = clf.predict(len_dv)
    f1 = f1_score(y_dv, pred, average="macro")
    baseline_rows.append({"slice": "length_only", "macro_F1": round(f1, 4), "AUC": float("nan"), "sec": 0})
    print("length_only macro-F1:", round(f1, 4))

    baseline_df = pd.DataFrame(baseline_rows)
    print()
    print(baseline_df.to_string(index=False))
    print("\npart_1..part_5 = fifths of the document in order (part_5 is the end).")
else:
    baseline_df = pd.DataFrame(baseline_rows)
    print("Skipped baselines (no dev docs or only one class in train).")

## 8. Load the Phase-1 tokenizer

In [ ]:
from transformers import AutoTokenizer
os.environ["TOKENIZERS_PARALLELISM"] = "true"


def find_tokenizer_dir():
    if TOKENIZER_DIR is not None:
        return TOKENIZER_DIR
    paths = glob.glob("/kaggle/input/**/tokenizer.json", recursive=True)
    folders = []
    for p in paths:
        folder = os.path.dirname(p)
        if "checkpoint-" not in folder:
            folders.append(folder)
    folders = sorted(set(folders))
    if len(folders) == 0:
        raise FileNotFoundError("No tokenizer.json under /kaggle/input. Attach your tokenizer dataset (Add Input).")
    for folder in folders:
        if os.path.exists(os.path.join(folder, "trained_on.json")):
            return folder
    return folders[0]


tok_dir = find_tokenizer_dir()
tokenizer = AutoTokenizer.from_pretrained(tok_dir)
tokenizer.model_max_length = 10 ** 9
VOCAB_SIZE = len(tokenizer)
print("Tokenizer:", tok_dir, "| vocab size:", VOCAB_SIZE)


def find_sep_id(tok):
    if tok.eos_token_id is not None:
        return int(tok.eos_token_id)
    for name in ["<eos>", "</s>", "<|endoftext|>", "<EOS>"]:
        tid = tok.convert_tokens_to_ids(name)
        if tid is not None and tid != tok.unk_token_id:
            return int(tid)
    raise ValueError("Could not find an EOS-like token to use as separator.")


SEP_ID = find_sep_id(tokenizer)
if tokenizer.pad_token_id is not None:
    PAD_ID = int(tokenizer.pad_token_id)
else:
    PAD_ID = SEP_ID
print("SEP (EOS) id:", SEP_ID, "| PAD id:", PAD_ID)

if VOCAB_SIZE < 65535:
    ID_DTYPE = np.uint16
else:
    ID_DTYPE = np.int32
print("ids stored as:", ID_DTYPE.__name__)

# Oracle check: for a byte-level BPE, decode(encode(text)) should give the text back.
sample_texts = []
for d in clean_docs["train"][:30]:
    sample_texts.append(d["text"])
ok = 0
for t in sample_texts:
    ids = tokenizer(t, add_special_tokens=False)["input_ids"]
    back = tokenizer.decode(ids, clean_up_tokenization_spaces=False)
    if back == t:
        ok += 1
print("round-trip exact match:", ok, "/", len(sample_texts))
if ok < len(sample_texts):
    print("Not exact: the tokenizer normalises or lowercases text. Fine as long as it is the same one Phase 1 used.")

## 9. Tokenise each document once (no truncation)

In [ ]:
def tokenize_texts(texts, batch_size=32, name=""):
    all_ids = []
    t0 = time.time()
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for b in range(n_batches):
        batch = texts[b * batch_size:(b + 1) * batch_size]
        out = tokenizer(batch, add_special_tokens=False)
        for ids in out["input_ids"]:
            all_ids.append(np.array(ids, dtype=ID_DTYPE))
        if b % 100 == 0:
            print(name, "batch", b + 1, "/", n_batches, "| elapsed", round(time.time() - t0, 1), "s")
    return all_ids


tokenized = {}
for split in SPLITS:
    texts = []
    for d in clean_docs[split]:
        texts.append(d["text"])
    tokenized[split] = tokenize_texts(texts, name=split)
    print(split, "tokenised:", len(tokenized[split]))

# sanity
max_id = 0
n_unk = 0
n_tokens_total = 0
unk_id = tokenizer.unk_token_id
for split in SPLITS:
    for arr in tokenized[split]:
        if len(arr) == 0:
            continue
        m = int(arr.max())
        if m > max_id:
            max_id = m
        n_tokens_total += len(arr)
        if unk_id is not None:
            n_unk += int((arr == unk_id).sum())
print("max token id:", max_id, "(vocab", VOCAB_SIZE, ")")
assert max_id < VOCAB_SIZE
print("total tokens:", n_tokens_total, "| unk tokens:", n_unk)

## 10. Token-length analysis and long-document views

In [ ]:
tok_rows = []
for split in SPLITS:
    for i in range(len(clean_docs[split])):
        tok_rows.append({
            "split": split,
            "id": str(clean_docs[split][i]["id"]),
            "label": clean_docs[split][i]["label"],
            "n_tokens": len(tokenized[split][i]),
        })
df_tok = pd.DataFrame(tok_rows)

print("=== Token length by split ===")
print(df_tok.groupby("split")["n_tokens"].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

lab_tok = df_tok[df_tok["split"].isin(["train", "dev"])]
print("\n=== Token length by label (train+dev) ===")
print(lab_tok.groupby("label")["n_tokens"].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

chars_per_token = sum(df_all["n_chars"]) / max(1, n_tokens_total)
print("\nchars per token:", round(chars_per_token, 2))

print("\n=== % of docs longer than L tokens (needs truncation / chunking) ===")
for L in [512, 1024, 2048, 4096, 8192, 16384]:
    line = "L=" + str(L).rjust(5) + " | "
    for split in SPLITS:
        sub = df_tok[df_tok["split"] == split]["n_tokens"]
        if len(sub) > 0:
            line += split + ": " + str(round(100.0 * (sub > L).mean(), 1)).rjust(5) + "%   "
    print(line)

print("\n=== Chunking summary (chunk_len=%d, overlap=%d) ===" % (CHUNK_LEN, CHUNK_OVERLAP))
for split in ["train", "dev", "test"]:
    counts = []
    for n in df_tok[df_tok["split"] == split]["n_tokens"]:
        starts = []
        body_len = CHUNK_LEN - 1
        stride = body_len - CHUNK_OVERLAP
        s = 0
        while True:
            starts.append(s)
            if s + body_len >= n:
                break
            s += stride
        counts.append(len(starts))
    if len(counts) > 0:
        counts = np.array(counts)
        print(split, "| mean chunks:", round(counts.mean(), 1), "| median:", int(np.median(counts)),
              "| p90:", int(np.percentile(counts, 90)), "| max:", int(counts.max()),
              "| pct over MAX_CHUNKS (" + str(MAX_CHUNKS) + "):", round(100.0 * (counts > MAX_CHUNKS).mean(), 1))

if lab_tok["label"].nunique() == 2:
    auc_tok = roc_auc_score(lab_tok["label"], lab_tok["n_tokens"])
    print("\nAUC of token length alone:", round(auc_tok, 3))

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# label balance
counts = lab_df["label"].value_counts().sort_index()
axes[0].bar([str(x) for x in counts.index], counts.values, color=["#4C72B0", "#DD8452"][:len(counts)])
axes[0].set_title("Label counts (train+dev)")
axes[0].set_xlabel("label")

# token length by label
for lab, color in [(0, "#4C72B0"), (1, "#DD8452")]:
    vals = lab_tok[lab_tok["label"] == lab]["n_tokens"].values
    if len(vals) > 0:
        axes[1].hist(np.log10(np.maximum(vals, 1)), bins=40, alpha=0.6, label="label " + str(lab), color=color)
axes[1].set_title("Tokens per doc (log10)")
axes[1].set_xlabel("log10(tokens)")
axes[1].legend()

# positional baseline
if len(baseline_df) > 0:
    axes[2].barh(baseline_df["slice"], baseline_df["macro_F1"], color="#55A868")
    axes[2].set_title("TF-IDF+LR macro-F1 by slice (dev)")
    axes[2].set_xlim(0, 1)
    axes[2].invert_yaxis()
else:
    axes[2].set_title("no baseline")

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "analysis.png")
plt.savefig(fig_path, dpi=120)
plt.show()
print("saved", fig_path)

## 11. Folds (train+dev combined) and class weights

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Row order: train docs, then dev docs. Fold is stratified by label and by length tercile.
fold_index = []     # (split, position)
fold_labels = []
fold_tokens = []
for split in ["train", "dev"]:
    for i in range(len(clean_docs[split])):
        fold_index.append((split, i))
        fold_labels.append(clean_docs[split][i]["label"])
        fold_tokens.append(len(tokenized[split][i]))
fold_labels = np.array(fold_labels)
fold_tokens = np.array(fold_tokens)

low = np.percentile(fold_tokens, 33.3)
high = np.percentile(fold_tokens, 66.6)
len_bin = np.zeros(len(fold_tokens), dtype=int)
len_bin[fold_tokens > low] = 1
len_bin[fold_tokens > high] = 2
strat_key = fold_labels * 3 + len_bin

key_counts = Counter(strat_key.tolist())
if min(key_counts.values()) < N_FOLDS:
    print("Some label x length cells are too small, stratifying by label only.")
    strat_key = fold_labels

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_of_row = np.full(len(fold_labels), -1, dtype=int)
for fold_id, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(strat_key)), strat_key)):
    fold_of_row[va_idx] = fold_id

fold_arrays = {
    "train": np.full(len(clean_docs["train"]), -1, dtype=int),
    "dev": np.full(len(clean_docs["dev"]), -1, dtype=int),
    "test": np.full(len(clean_docs["test"]), -1, dtype=int),
}
for row in range(len(fold_index)):
    split, i = fold_index[row]
    fold_arrays[split][i] = fold_of_row[row]

print("fold sizes and label-1 share:")
for f in range(N_FOLDS):
    mask = fold_of_row == f
    print("  fold", f, "| docs:", int(mask.sum()), "| label1 %:", round(100.0 * fold_labels[mask].mean(), 1),
          "| mean tokens:", int(fold_tokens[mask].mean()))

# class weights from train+dev
n_total = len(fold_labels)
count_per_class = [int((fold_labels == 0).sum()), int((fold_labels == 1).sum())]


def normalise_weights(w, counts):
    total = 0.0
    for k in range(2):
        total += w[k] * counts[k]
    scale = n_total / total
    out = []
    for k in range(2):
        out.append(float(w[k] * scale))
    return out


w_inverse = []
w_sqrt = []
w_eff = []
beta = 0.999
for k in range(2):
    c = max(1, count_per_class[k])
    w_inverse.append(1.0 / c)
    w_sqrt.append(1.0 / np.sqrt(c))
    w_eff.append((1.0 - beta) / (1.0 - beta ** c))

class_weights = {
    "counts": count_per_class,
    "inverse": normalise_weights(w_inverse, count_per_class),
    "inverse_sqrt": normalise_weights(w_sqrt, count_per_class),
    "effective_num": normalise_weights(w_eff, count_per_class),
    "bce_pos_weight": float(count_per_class[0] / max(1, count_per_class[1])),
}
print("\nclass weights:", json.dumps(class_weights, indent=1))

## 12. Helpers saved for the training notebooks

In [ ]:
%%writefile cjpe_utils.py

import json
import numpy as np


def head_tail_ids(ids, max_len, sep_id, head_frac=0.5):
    # Keep the first part and the last part of a long document.
    # ids = 1D token ids without special tokens. Output length <= max_len, ends with sep_id.
    ids = np.asarray(ids)
    body_len = max_len - 1
    n = len(ids)
    if n <= body_len:
        out = ids.tolist()
        out.append(int(sep_id))
        return out
    head_n = int(body_len * head_frac)
    tail_n = body_len - head_n
    out = []
    if head_n > 0:
        out.extend(ids[:head_n].tolist())
    if tail_n > 0:
        out.extend(ids[n - tail_n:].tolist())
    out.append(int(sep_id))
    return out


def chunk_starts(n_tokens, chunk_len, overlap):
    # Start offsets of overlapping chunks that cover the whole document.
    body_len = chunk_len - 1
    stride = body_len - overlap
    if stride < 1:
        raise ValueError("overlap must be smaller than chunk_len - 1")
    starts = []
    start = 0
    while True:
        starts.append(start)
        if start + body_len >= n_tokens:
            break
        start = start + stride
    return starts


def pick_starts(starts, max_chunks, pick):
    # Reduce too many chunks to max_chunks.
    # pick: first | last | head_tail | uniform
    total = len(starts)
    if max_chunks is None or total <= max_chunks:
        return starts
    if pick == "first":
        return starts[:max_chunks]
    if pick == "last":
        return starts[total - max_chunks:]
    if pick == "head_tail":
        n_head = max_chunks // 2
        n_tail = max_chunks - n_head
        return starts[:n_head] + starts[total - n_tail:]
    if pick == "uniform":
        positions = np.linspace(0, total - 1, max_chunks)
        chosen = []
        for p in positions:
            idx = int(round(float(p)))
            if idx not in chosen:
                chosen.append(idx)
        result = []
        for idx in chosen:
            result.append(starts[idx])
        return result
    raise ValueError("unknown pick: " + str(pick))


def make_chunks(ids, sep_id, chunk_len, overlap, max_chunks=None, pick="uniform"):
    # Returns (chunks, rel_pos). Each chunk ends with sep_id.
    # rel_pos = where the chunk starts inside the document, 0..1 (use as a position feature).
    ids = np.asarray(ids)
    n = len(ids)
    body_len = chunk_len - 1
    starts = chunk_starts(n, chunk_len, overlap)
    starts = pick_starts(starts, max_chunks, pick)
    chunks = []
    rel_pos = []
    for s in starts:
        piece = ids[s:s + body_len].tolist()
        piece.append(int(sep_id))
        chunks.append(piece)
        rel_pos.append(s / max(1, n))
    return chunks, rel_pos


def pad_right(seqs, pad_id):
    # Right-pad a list of id lists. Returns input_ids [B, T], attention_mask [B, T], lengths [B].
    lengths = []
    for s in seqs:
        lengths.append(len(s))
    max_len = max(lengths)
    batch = len(seqs)
    input_ids = np.full((batch, max_len), int(pad_id), dtype=np.int64)
    mask = np.zeros((batch, max_len), dtype=np.int64)
    for i in range(batch):
        L = lengths[i]
        input_ids[i, :L] = np.asarray(seqs[i], dtype=np.int64)
        mask[i, :L] = 1
    return input_ids, mask, np.asarray(lengths, dtype=np.int64)


def macro_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    f1_list = []
    for c in [0, 1]:
        tp = int(((y_true == c) & (y_pred == c)).sum())
        fp = int(((y_true != c) & (y_pred == c)).sum())
        fn = int(((y_true == c) & (y_pred != c)).sum())
        if tp == 0:
            f1_list.append(0.0)
        else:
            precision = tp / (tp + fp)
            recall = tp / (tp + fn)
            f1_list.append(2 * precision * recall / (precision + recall))
    return float(sum(f1_list) / 2.0)


def best_threshold(probs_1, y_true):
    # Scan thresholds on P(label=1) and return (best_threshold, best_macro_f1).
    probs_1 = np.asarray(probs_1)
    best_t = 0.5
    best_f = -1.0
    for t in np.arange(0.05, 0.951, 0.01):
        preds = (probs_1 >= t).astype(int)
        f = macro_f1(y_true, preds)
        if f > best_f:
            best_f = f
            best_t = float(t)
    return best_t, best_f


def write_submission(doc_ids, preds, path):
    # Codabench format: JSON list of {"id": ..., "pred": 0 or 1}, one per test case.
    if len(doc_ids) != len(preds):
        raise ValueError("doc_ids and preds differ in length")
    rows = []
    for i in range(len(doc_ids)):
        p = int(preds[i])
        if p not in (0, 1):
            raise ValueError("pred must be 0 or 1")
        rows.append({"id": doc_ids[i], "pred": p})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f)
    print("wrote", len(rows), "predictions to", path)

In [ ]:
sys.path.insert(0, os.getcwd())
import importlib
import cjpe_utils
importlib.reload(cjpe_utils)
shutil.copy("cjpe_utils.py", os.path.join(OUT_DIR, "cjpe_utils.py"))

# self-check of the views on real documents
checked = 0
for split in ["train", "test"]:
    n_check = min(40, len(tokenized[split]))
    for i in range(n_check):
        ids = tokenized[split][i]
        n = len(ids)

        ht = cjpe_utils.head_tail_ids(ids, HEAD_TAIL_LEN, SEP_ID, HEAD_FRAC)
        assert len(ht) <= HEAD_TAIL_LEN
        assert ht[-1] == SEP_ID
        if n <= HEAD_TAIL_LEN - 1:
            assert ht[:-1] == ids.tolist()

        chunks, rel = cjpe_utils.make_chunks(ids, SEP_ID, CHUNK_LEN, CHUNK_OVERLAP, None, "uniform")
        covered = np.zeros(max(n, 1), dtype=bool)
        stride = CHUNK_LEN - 1 - CHUNK_OVERLAP
        for k in range(len(chunks)):
            assert chunks[k][-1] == SEP_ID
            assert len(chunks[k]) <= CHUNK_LEN
            start = k * stride
            body = chunks[k][:-1]
            assert body == ids[start:start + len(body)].tolist()
            covered[start:start + len(body)] = True
        if n > 0:
            assert covered.all()

        capped, rel2 = cjpe_utils.make_chunks(ids, SEP_ID, CHUNK_LEN, CHUNK_OVERLAP, MAX_CHUNKS, "head_tail")
        assert len(capped) <= MAX_CHUNKS
        checked += 1
print("view self-check passed on", checked, "docs")

batch_ids, batch_mask, batch_len = cjpe_utils.pad_right([[1, 2, 3], [4, 5]], PAD_ID)
assert batch_ids.shape == (2, 3) and batch_mask.sum() == 5
print("pad_right ok")

## 13. Save

In [ ]:
for split in SPLITS:
    folder = "Train" if split in ["train", "dev"] else "Test"
    folder_path = os.path.join(OUT_DIR, folder)
    os.makedirs(folder_path, exist_ok=True)
    rows_out = []
    for d in clean_docs[split]:
        row = {"id": d["id"], "text": d["text"]}
        if split != "test":
            row["label"] = d["label"]
        rows_out.append(row)
    with open(os.path.join(folder_path, split + ".json"), "w", encoding="utf-8") as f:
        json.dump(rows_out, f, ensure_ascii=False)

bundle = {
    "vocab_size": VOCAB_SIZE,
    "sep_id": SEP_ID,
    "pad_id": PAD_ID,
    "id_dtype": ID_DTYPE.__name__,
    "tokenizer_dir": tok_dir,
    "class_weights": class_weights,
    "config": {
        "seed": SEED, "n_folds": N_FOLDS, "chunk_len": CHUNK_LEN, "chunk_overlap": CHUNK_OVERLAP,
        "max_chunks": MAX_CHUNKS, "head_tail_len": HEAD_TAIL_LEN, "head_frac": HEAD_FRAC,
        "keep_newlines": KEEP_NEWLINES,
    },
    "splits": {},
}
for split in SPLITS:
    labels = None
    if split != "test":
        labels = np.array([d["label"] for d in clean_docs[split]], dtype=np.int64)
    bundle["splits"][split] = {
        "doc_ids": [d["id"] for d in clean_docs[split]],
        "ids": tokenized[split],
        "labels": labels,
        "n_chars": np.array([len(d["text"]) for d in clean_docs[split]], dtype=np.int64),
        "fold": fold_arrays[split],
    }

with open(os.path.join(OUT_DIR, "tokenized_cjpe.pkl"), "wb") as f:
    pickle.dump(bundle, f, protocol=4)

with open(os.path.join(OUT_DIR, "class_weights.json"), "w") as f:
    json.dump(class_weights, f, indent=1)
with open(os.path.join(OUT_DIR, "audit_report.json"), "w") as f:
    json.dump(audit, f, indent=1)
baseline_df.to_csv(os.path.join(OUT_DIR, "slice_baselines.csv"), index=False)
df_tok.to_csv(os.path.join(OUT_DIR, "doc_lengths.csv"), index=False)

meta = {
    "vocab_size": VOCAB_SIZE, "sep_id": SEP_ID, "pad_id": PAD_ID,
    "counts": {s: len(clean_docs[s]) for s in SPLITS},
    "total_tokens": int(n_tokens_total),
}
with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=1)

for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print(name.ljust(26), round(os.path.getsize(full) / 1e6, 1), "MB")

## 14. Verify the saved files

In [ ]:
with open(os.path.join(OUT_DIR, "tokenized_cjpe.pkl"), "rb") as f:
    check = pickle.load(f)

for split in SPLITS:
    block = check["splits"][split]
    n = len(block["doc_ids"])
    assert len(block["ids"]) == n
    with open(os.path.join(OUT_DIR, "Train" if split != "test" else "Test", split + ".json"), "r", encoding="utf-8") as f:
        saved_rows = json.load(f)
    assert len(saved_rows) == n
    if split != "test":
        assert len(block["labels"]) == n
        assert set(block["labels"].tolist()).issubset({0, 1})
    rng2 = np.random.RandomState(0)
    n_pick = min(15, n)
    if n_pick > 0:
        picks = rng2.choice(n, n_pick, replace=False)
        for i in picks:
            i = int(i)
            again = tokenizer(saved_rows[i]["text"], add_special_tokens=False)["input_ids"]
            assert again == block["ids"][i].tolist()
            assert str(saved_rows[i]["id"]) == str(block["doc_ids"][i])
    print(split, "OK |", n, "docs")

# test ids must match what goes into the submission
print("\nTest ids (first 3):", check["splits"]["test"]["doc_ids"][:3])
print("Label sets:", {s: sorted(set(check["splits"][s]["labels"].tolist())) for s in ["train", "dev"] if len(check["splits"][s]["labels"]) > 0})
print("\nALL CHECKS PASSED")

## 15. How later notebooks load this

```python
import pickle, sys
DATA = "/kaggle/input/<your-saved-output-dataset>"   # or /kaggle/working/cjpe_data_clean
sys.path.insert(0, DATA)
import cjpe_utils
with open(DATA + "/tokenized_cjpe.pkl", "rb") as f:
    data = pickle.load(f)
SEP_ID, PAD_ID = data["sep_id"], data["pad_id"]
tr = data["splits"]["train"]       # doc_ids, ids (np arrays), labels, fold
ids = tr["ids"][0].astype("int64")
view = cjpe_utils.head_tail_ids(ids, 1024, SEP_ID, 0.5)
chunks, rel_pos = cjpe_utils.make_chunks(ids, SEP_ID, 1024, 128, 16, "uniform")
```
Folds: `fold` is 0..4 for train and dev rows (train+dev pooled, stratified); -1 for test.
Submission: `cjpe_utils.write_submission(test_doc_ids, preds, "prediction.json")`.